In [ ]:
# Function Tools: Building Custom Actions
# Welcome! In this notebook, you’ll learn how to create and integrate custom tools in Google’s Agent Development Kit (ADK).  

In [3]:
# Install Google ADK for Python
%pip install google-adk --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# (optional) Verify installation
%pip show google-adk

Name: google-adk
Version: 1.0.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: authlib, click, fastapi, google-api-python-client, google-cloud-aiplatform, google-cloud-secret-manager, google-cloud-speech, google-cloud-storage, google-genai, graphviz, mcp, opentelemetry-api, opentelemetry-exporter-gcp-trace, opentelemetry-sdk, pydantic, python-dotenv, PyYAML, sqlalchemy, tzlocal, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [5]:
# Configure environment
import os
# We use Gemini as our language model and ensure we are not using Vertex AI for this demo.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

In [7]:
# Import Required Modules
import requests
from typing import Dict, Any, Optional
from datetime import datetime, timedelta
from collections import Counter
from dateutil import parser as date_parser
import calendar
import re

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from IPython.display import Markdown, display

In [8]:
# OpenWeather API: Set Up
OPEN_WEATHER_API_KEY = os.environ.get("OPEN_WEATHER_API_KEY")
if not OPEN_WEATHER_API_KEY:
    raise ValueError("OPEN_WEATHER_API_KEY environment variable not set.  Please set it.")

In [9]:
# Helper Functions: Parse flexible date expressions for weather queries

def parse_flexible_date_range(date_str: str) -> Optional[tuple]:
    """
    Parses a wide variety of date expressions and returns (start_date, end_date).
    Supports:
    - 'today', 'tomorrow', 'in 3 days'
    - 'this weekend', 'next weekend'
    - 'next week', 'this week'
    - 'first week of July', 'second week of August', etc.
    - Specific dates: '2025-07-01', 'July 1', '07-01'
    Returns (datetime, datetime) or None if parsing fails.
    """
    date_str = date_str.lower().strip()
    today = datetime.today()
    weekday = today.weekday()  # Monday=0, Sunday=6

    # Today, tomorrow, in X days
    if date_str in ["today", "now"]:
        return today, today
    if date_str == "tomorrow":
        tmr = today + timedelta(days=1)
        return tmr, tmr
    match = re.match(r"in (\d+) days?", date_str)
    if match:
        days = int(match.group(1))
        target = today + timedelta(days=days)
        return target, target

    # This week (Monday to Sunday)
    if date_str == "this week":
        start = today - timedelta(days=weekday)
        end = start + timedelta(days=6)
        return start, end

    # Next week (Monday to Sunday)
    if date_str == "next week":
        start = today - timedelta(days=weekday) + timedelta(days=7)
        end = start + timedelta(days=6)
        return start, end

    # This weekend (Saturday & Sunday)
    if date_str == "this weekend":
        saturday = today + timedelta((5 - weekday) % 7)
        sunday = saturday + timedelta(days=1)
        return saturday, sunday

    # Next weekend (Saturday & Sunday)
    if date_str == "next weekend":
        saturday = today + timedelta((5 - weekday) % 7 + 7)
        sunday = saturday + timedelta(days=1)
        return saturday, sunday

    # "first week of July", "second week of August", etc.
    match = re.match(r"(first|second|third|fourth|last) week of (\w+)", date_str)
    if match:
        week_map = {
            "first": 0, "second": 1, "third": 2, "fourth": 3, "last": -1
        }
        week_num = week_map[match.group(1)]
        month_str = match.group(2)
        try:
            month = list(calendar.month_name).index(month_str.capitalize())
            year = today.year
            # If the month has already passed, assume next year
            if month < today.month:
                year += 1
            cal = calendar.monthcalendar(year, month)
            if week_num == -1:
                week = cal[-1]
            else:
                week = cal[week_num]
            # Get Monday (0) and Sunday (6) of that week
            start_day = week[0] if week[0] != 0 else week[calendar.SATURDAY]
            end_day = week[6] if week[6] != 0 else week[calendar.FRIDAY]
            start = datetime(year, month, start_day)
            # Find first non-zero day for start, last non-zero for end
            for d in week:
                if d != 0:
                    start = datetime(year, month, d)
                    break
            for d in reversed(week):
                if d != 0:
                    end = datetime(year, month, d)
                    break
            return start, end
        except Exception:
            return None

    # Try parsing as a specific date or date range
    try:
        # "July 1", "07-01", "2025-07-01"
        dt = date_parser.parse(date_str, fuzzy=True, default=today)
        return dt, dt
    except Exception:
        pass

    # "July 1 to July 5", "07-01 to 07-05"
    if " to " in date_str:
        parts = date_str.split(" to ")
        try:
            start = date_parser.parse(parts[0], fuzzy=True, default=today)
            end = date_parser.parse(parts[1], fuzzy=True, default=today)
            return start, end
        except Exception:
            return None

    return None


In [10]:
# Custom Tool: Get Current weather
def get_current_weather_from_openweather(city: str) -> Dict[str, Any]:
    """
    Retrieves current weather data for a given city.
    """
    print(f"--- Tool: get_current_weather called for city: {city} ---")
    try:
        url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPEN_WEATHER_API_KEY}&units=metric"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        if data["cod"] == 200:
            description = data["weather"][0]["description"]
            temperature = data["main"]["temp"]
            humidity = data["main"]["humidity"]
            wind_speed = data["wind"]["speed"]
            report = (
                f"The weather in {city} is {description} with a temperature of {temperature}°C. "
                f"Humidity is {humidity}% and wind speed is {wind_speed} m/s."
            )
            return {"status": "success", "report": report}
        else:
            return {"status": "error", "error_message": f"OpenWeatherMap API Error: {data['message']}"}
    except requests.exceptions.RequestException as e:
        return {"status": "error", "error_message": f"Error fetching weather data: {e}"}
    except KeyError as e:
        return {"status": "error", "error_message": f"Error parsing weather data: Missing key: {e}"}
    except Exception as e:
        return {"status": "error", "error_message": f"An unexpected error occurred: {e}"}


In [11]:
# Custom Tool: Get weather summary for a flexible date range
def get_weather_summary_from_openweather(city: str, date_expr: str) -> Dict[str, Any]:
    """
    Summarizes weather data for a given city and flexible date expression.
    """
    print(f"--- Tool: get_weather_summary called for city: {city}, date_expr: {date_expr} ---")
    try:
        date_range = parse_flexible_date_range(date_expr)
        if not date_range:
            return {"status": "error", "error_message": f"Could not parse date expression: '{date_expr}'"}
        start, end = date_range
        num_days = (end - start).days + 1

        # Use OpenWeatherMap's 5-day forecast API
        url = f"http://api.openweathermap.org/data/2.5/forecast?q={city}&appid={OPEN_WEATHER_API_KEY}&units=metric"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Get current weather as fallback
        current_url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPEN_WEATHER_API_KEY}&units=metric"
        current_response = requests.get(current_url)
        current_response.raise_for_status()
        current_data = current_response.json()

        all_temps = []
        all_descriptions = []
        for entry in data.get("list", []):
            entry_date = datetime.fromtimestamp(entry["dt"])
            if start <= entry_date <= end:
                all_temps.append(entry["main"]["temp"])
                all_descriptions.append(entry["weather"][0]["description"])

        # If no forecast entries match our date range, use current weather as fallback
        if not all_temps:
            all_temps = [current_data["main"]["temp"]]
            all_descriptions = [current_data["weather"][0]["description"]]

        min_temp = round(min(all_temps), 1)
        max_temp = round(max(all_temps), 1)
        avg_temp = round(sum(all_temps) / len(all_temps), 1)
        weather_types = [desc for desc, _ in Counter(all_descriptions).most_common(3)]

        weather_summary = {
            "destination": data["city"]["name"],
            "country": data["city"]["country"],
            "trip_length": num_days,
            "date_range": f"{start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}",
            "avg_temp": avg_temp,
            "temp_range": f"{min_temp}°C to {max_temp}°C",
            "weather_types": weather_types
        }
        return weather_summary
    except requests.exceptions.RequestException as e:
        return {"status": "error", "error_message": f"Error fetching weather data: {e}"}
    except KeyError as e:
        return {"status": "error", "error_message": f"Error parsing weather data: {e}"}
    except Exception as e:
        return {"status": "error", "error_message": f"An unexpected error occurred: {e}"}


In [12]:
weather_agent = Agent(
    name="weather_agent_v1",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Provides weather information for specific cities and flexible date expressions.",
    instruction=(
        "You are a helpful weather assistant. "
        "When the user asks for the current weather in a specific city, "
        "use the 'get_current_weather_from_openweather' tool. "
        "When the user asks for weather for a specific date or date range, use the 'get_weather_summary_from_openweather' tool. "
        "The 'get_weather_summary_from_openweather' tool can handle flexible date expressions like 'this weekend', 'next week', 'first week of July', as well as specific dates. "
        "If a tool returns an error, inform the user politely. "
        "If a tool is successful, present the weather report clearly."
    ),
    tools=[get_current_weather_from_openweather, get_weather_summary_from_openweather],
)

In [13]:
# Set up the session and user input
APP_NAME = "wanderwise_app"
USER_ID = "user_001"
SESSION_ID = "weather_demo_session"

session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

runner = Runner(
    agent=weather_agent,
    app_name=APP_NAME,
    session_service=session_service
)

In [15]:
# Helper function to run and display agent responses
def ask_weather_agent(prompt):
    content = types.Content(role="user", parts=[types.Part(text=prompt)])
    events = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )
    for event in events:
        if event.is_final_response():
            display(Markdown(f"**Prompt:** {prompt}\n\n---\n\n**Agent Response:**\n\n{event.content.parts[0].text}"))

In [16]:
# Example queries to showcase flexible date parsing

# Weather in London next week
ask_weather_agent("What will the weather be like in London next week?")

# Paris this weekend
ask_weather_agent("What's the weather in Paris this weekend?")

# Tokyo in July first week
ask_weather_agent("How's the weather in Tokyo in the first week of July?")

# Rome tomorrow
ask_weather_agent("Will it rain in Rome tomorrow?")

# New York in 3 days
ask_weather_agent("Weather forecast for New York in 3 days?")

# Mumbai July 1 to July 5
ask_weather_agent("What will the weather be like in Mumbai July 1 to July 5?")


--- Tool: get_weather_summary called for city: London, date_expr: next week ---


**Prompt:** What will the weather be like in London next week?

---

**Agent Response:**

The weather in London next week (June 2nd to June 8th, 2025) will be characterized by few clouds, with an average temperature of 11.6°C, ranging from 11.6°C to 11.6°C.

--- Tool: get_weather_summary called for city: Paris, date_expr: this weekend ---


**Prompt:** What's the weather in Paris this weekend?

---

**Agent Response:**

This weekend (May 31st to June 1st, 2025) in Paris, the weather will be clear sky with an average temperature of 10.7°C, ranging from 10.7°C to 10.7°C.

--- Tool: get_weather_summary called for city: Tokyo, date_expr: first week of July ---


...

**Prompt:** Will it rain in Rome tomorrow?

---

**Agent Response:**

The weather in Rome tomorrow (May 27th, 2025) will be broken clouds with an average temperature of 19.0°C. There is no rain predicted.

--- Tool: get_weather_summary called for city: New York, date_expr: in 3 days ---


**Prompt:** Weather forecast for New York in 3 days?

---

**Agent Response:**

The weather in New York in 3 days (May 29th, 2025) will be few clouds with an average temperature of 15.7°C.

--- Tool: get_weather_summary called for city: Mumbai, date_expr: July 1 to July 5 ---


**Prompt:** What will the weather be like in Mumbai July 1 to July 5?

---

**Agent Response:**

The weather in Mumbai from July 1st to July 5th, 2025 will be thunderstorm with heavy rain, with an average temperature of 28.0°C.

In [ ]:
# 🎉 Congratulations!
# You now have a flexible, production-ready ADK agent that can handle a wide range of natural date expressions for weather queries.  
# Try more prompts, or extend your agent with additional travel tools!